# QMS 가상 데이터 시드 노트북

Mock MES를 실시간 조회해 그와 맞물리는 QMS 데이터 1,004행(8테이블)을 만들고
이 레이크하우스에 Delta 테이블로 적재합니다.

## 연동 원칙

MES와 QMS는 서로 다른 시스템입니다. DB 수준의 외래키는 없고
`lot_id`, `product_code`, `step_code`, `material_code` 라는 비즈니스 키로만 이어집니다.
같은 사실을 양쪽이 중복해서 갖지 않습니다. QMS에는 `mes_result`, `scrap_qty`,
`operator`, `in_qty`, `out_qty` 컬럼이 없습니다. 생산 결과를 알고 싶으면 MES에 물어야 합니다.

## 실행 순서

1. 오른쪽 패널에서 **대상 레이크하우스를 Attach** 합니다. 적재 위치는 이걸로 정해집니다.
2. 파라미터 셀에서 MES API 키 조달 방법을 정합니다.
3. 전체 실행합니다.

고정 시드와 `overwrite` 모드를 쓰므로 몇 번을 다시 돌려도 결과가 같습니다.


In [ ]:
# Fabric 파이프라인에서 이 셀의 값을 덮어쓸 수 있습니다.
# 적재 대상은 이 노트북에 Attach 한 레이크하우스입니다. 이름 문자열로 정하지 않습니다.
# 스키마 사용(schema-enabled) 레이크하우스에 넣을 때만 TARGET_SCHEMA 를 채우세요. 예: "dbo"
TARGET_SCHEMA = ""
MES_BASE_URL = "https://mock-mes.greenrock-bb44c93a.koreacentral.azurecontainerapps.io"

# MES 인증 키. 이 작업 영역은 공유될 수 있고 노트북은 자동 저장되니 실행 후 지우세요.
MES_API_KEY = ""

TABLE_PREFIX = "qms_"
WRITE_MODE = "overwrite"


In [ ]:
"""QMS 도메인 참조 상수.

MES에서 오지 않는 값은 전부 여기에 있다. 생성 모듈은 이 상수와 MesSnapshot만
읽고 동작하므로, 값을 바꾸면 결과가 어떻게 달라지는지 한곳에서 파악된다.
"""

from __future__ import annotations

import datetime as dt

# 기준일 상수를 두지 않는다. QMS 의 모든 시각은 MES 앵커에서 유도되며
# mes_client.anchor_date(snapshot) 가 그 역할을 한다. 벽시계 상수를 여기 두면
# MES 재배포 때 QMS 만 제자리에 남는다.

# 모듈마다 독립 시드를 둔다. 한 모듈의 난수 소비량이 바뀌어도
# 다른 모듈의 출력이 흔들리지 않게 하기 위함이다.
SEED_MASTERS = 20260904
SEED_INSPECTION = 20260905
SEED_MEASUREMENT = 20260906
SEED_INCOMING = 20260907
SEED_NCR = 20260908

MES_DEFECT_CODES = (
    "Particle",
    "Scratch",
    "Overlay",
    "Etch-Residue",
    "Contamination",
    "CD-OOS",
)

SEVERITY_SCORE = {"Critical": 9, "Major": 6, "Minor": 3}

# (mes_defect_code, 코드 접두, defect_category, 주 발생 공정 CSV)
DEFECT_TAXONOMY = (
    ("Particle", "PTC", "오염", "DIFF,CVD,CMP"),
    ("Scratch", "SCR", "외관", "CMP,PKG"),
    ("Overlay", "OVL", "패턴", "PHOTO,METRO"),
    ("Etch-Residue", "ETR", "막질", "ETCH,CVD"),
    ("Contamination", "CTM", "오염", "DIFF,IMPL,CVD"),
    ("CD-OOS", "CDO", "치수", "PHOTO,ETCH,METRO"),
)

# (mes_defect_code, 순번, 한글명, 영문명, severity, 표준원인, 표준조치)
DEFECT_DETAILS = (
    ("Particle", 1, "파티클 오염 0.12um 이상", "Particle contamination over 0.12um", "Critical", "챔버 내벽 박리물 낙하", "챔버 습식세정 및 시즈닝 재실시"),
    ("Particle", 2, "장비 유래 파티클", "Equipment-borne particle", "Major", "이송 로봇 마모 분진", "로봇 암 교체 및 파티클 카운트 재측정"),
    ("Particle", 3, "가스라인 유래 파티클", "Gas line particle", "Major", "가스 필터 수명 초과", "인라인 필터 교체 및 퍼지"),
    ("Particle", 4, "인체 유래 파티클", "Human-borne particle", "Minor", "방진복 착용 절차 미준수", "클린룸 입실 교육 및 에어샤워 점검"),
    ("Scratch", 1, "CMP 연마 스크래치", "CMP polish scratch", "Critical", "슬러리 내 응집 입자", "슬러리 필터 교체 및 유량 재설정"),
    ("Scratch", 2, "웨이퍼 핸들링 스크래치", "Wafer handling scratch", "Major", "척 표면 이물", "척 세정 및 진공압 점검"),
    ("Scratch", 3, "캐리어 접촉 흠집", "Carrier contact mark", "Minor", "FOUP 슬롯 변형", "FOUP 교체 및 정렬 보정"),
    ("Scratch", 4, "패키지 표면 손상", "Package surface damage", "Minor", "몰드 이형 불량", "이형제 도포량 조정"),
    ("Overlay", 1, "정렬도 X 방향 초과", "Overlay X out of tolerance", "Critical", "스테이지 열변형", "스캐너 열보정 및 재정렬"),
    ("Overlay", 2, "정렬도 Y 방향 초과", "Overlay Y out of tolerance", "Critical", "레티클 장착 편차", "레티클 재장착 및 얼라인 재수행"),
    ("Overlay", 3, "회전 성분 편차", "Rotation component deviation", "Major", "웨이퍼 노치 정렬 오차", "노치 얼라이너 캘리브레이션"),
    ("Overlay", 4, "배율 성분 편차", "Magnification deviation", "Minor", "렌즈 온도 드리프트", "렌즈 온도 제어 루프 재조정"),
    ("Etch-Residue", 1, "폴리머 잔류물", "Polymer residue", "Critical", "에천트 조성 이탈", "가스 유량비 재설정 및 챔버 컨디셔닝"),
    ("Etch-Residue", 2, "금속 잔류물", "Metal residue", "Major", "오버에치 시간 부족", "에치 타임 연장 및 EPD 신호 재검토"),
    ("Etch-Residue", 3, "하드마스크 잔류", "Hard mask residue", "Major", "스트립 공정 누락", "애싱 레시피 보완"),
    ("Etch-Residue", 4, "측벽 잔류물", "Sidewall residue", "Minor", "패시베이션 과다", "패시베이션 가스 비율 하향"),
    ("Contamination", 1, "금속 오염 Cu", "Metallic contamination Cu", "Critical", "금속 배선 공정 교차 오염", "전용 챔버 분리 및 웨이퍼 세정"),
    ("Contamination", 2, "유기물 오염", "Organic contamination", "Major", "포토레지스트 잔류", "UV 오존 세정 추가"),
    ("Contamination", 3, "수분 오염", "Moisture contamination", "Major", "로드락 퍼지 부족", "퍼지 시간 연장 및 진공도 확인"),
    ("Contamination", 4, "이온성 오염", "Ionic contamination", "Minor", "초순수 비저항 저하", "UPW 라인 재생 및 수질 재측정"),
    ("CD-OOS", 1, "선폭 상한 초과", "CD above upper limit", "Critical", "노광량 부족", "도즈 재설정 및 FEM 재평가"),
    ("CD-OOS", 2, "선폭 하한 미달", "CD below lower limit", "Critical", "현상 시간 과다", "현상 레시피 시간 단축"),
    ("CD-OOS", 3, "선폭 균일도 이탈", "CD uniformity out of spec", "Major", "핫플레이트 온도 편차", "베이크 플레이트 존별 온도 보정"),
    ("CD-OOS", 4, "라인 에지 러프니스", "Line edge roughness", "Minor", "레지스트 감도 편차", "레지스트 로트 교체 및 재평가"),
)

# characteristic_code -> (한글명, measurement_type, unit, target, lsl, usl)
CHARACTERISTIC_BASE = {
    "CD": ("선폭", "계량형", "nm", 45.0, 40.5, 49.5),
    "OVL": ("정렬도", "계량형", "nm", 2.0, 0.0, 4.0),
    "THK": ("막두께", "계량형", "um", 1.20, 1.10, 1.30),
    "PTC": ("파티클수", "계수형", "ea", 8.0, 0.0, 20.0),
    "RS": ("면저항", "계량형", "Ω·sq", 120.0, 108.0, 132.0),
    "WRP": ("휨", "계량형", "%", 0.35, 0.05, 0.80),
}

# 공정별 관리 특성 3종. 4제품 × 9공정 × 3특성 = 108 검사기준.
STEP_CHARACTERISTICS = {
    "DIFF": ("THK", "PTC", "RS"),
    "PHOTO": ("CD", "OVL", "PTC"),
    "ETCH": ("CD", "THK", "PTC"),
    "IMPL": ("RS", "PTC", "THK"),
    "CVD": ("THK", "PTC", "RS"),
    "CMP": ("THK", "WRP", "PTC"),
    "METRO": ("CD", "OVL", "THK"),
    "TEST": ("RS", "CD", "PTC"),
    "PKG": ("WRP", "PTC", "THK"),
}

# 미세 노드일수록 선폭 규격이 좁다. CD 특성에만 적용한다.
PRODUCT_CD_SCALE = {"LX9": 0.55, "DDR5": 1.00, "NAND": 1.60, "PMIC": 2.40}

# QMS 계측기. MES 생산설비(EQP-*)와 완전히 별개 자산이다.
METROLOGY_EQP = {
    "CD": "MET-CD01",
    "OVL": "MET-OVL01",
    "THK": "MET-THK01",
    "PTC": "MET-PTC01",
    "RS": "MET-RS01",
    "WRP": "MET-WRP01",
}

SAMPLING_METHODS = {
    "계량형": ("5매 랜덤 9포인트", 5),
    "계수형": ("3매 전면 스캔", 3),
}

INSPECTION_FREQUENCIES = {
    "CD": "로트별",
    "OVL": "로트별",
    "THK": "로트별",
    "PTC": "전수",
    "RS": "시간별",
    "WRP": "시간별",
}

CONTROL_METHODS = {
    "계량형": "X-bar R 관리도",
    "계수형": "u 관리도",
}

# (팀명, 자격 보유 특성 CSV)
INSPECTOR_TEAMS = (
    ("계측팀", "CD,OVL,THK"),
    ("입고검사팀", "PTC,THK"),
    ("신뢰성팀", "RS,WRP"),
    ("출하검사팀", "CD,RS,PTC"),
    ("품질보증팀", "CD,OVL,THK,PTC,RS,WRP"),
)

# MES operator(kim.js 등)와 겹치지 않는 별도 인력 15명.
INSPECTOR_NAMES = (
    "강민우", "노현서", "서지훈", "오세영", "유다은",
    "임채원", "한도윤", "홍서아", "문가온", "배준호",
    "신예린", "안태경", "윤소민", "조하람", "하시우",
)

QUALIFICATION_LEVELS = ("초급", "중급", "선임", "책임")

# (supplier_code, supplier_name_ko)
SUPPLIERS = (
    ("SUP-A01", "한빛머티리얼즈"),
    ("SUP-A02", "동방정밀소재"),
    ("SUP-B01", "세종케미칼"),
    ("SUP-B02", "대륙화학"),
    ("SUP-C01", "성진가스"),
    ("SUP-C02", "에어프로덕트코리아"),
    ("SUP-D01", "태성메탈"),
    ("SUP-D02", "글로벌타겟"),
)

INSPECTION_ITEMS = {
    "Raw Wafer": "평탄도/저항률/외관",
    "Chemical": "순도/입도/비중",
    "Gas": "순도/수분/파티클",
    "Metal": "조성비/밀도/외관",
    "Mask": "CD 정확도/결함수/투과율",
    "Package": "치수/접합강도/외관",
}

ROOT_CAUSE_CATEGORIES = ("설비", "자재", "작업방법", "환경", "측정")

OWNER_DEPTS = ("공정기술팀", "설비기술팀", "자재구매팀", "품질보증팀", "생산관리팀")

OWNER_NAMES = ("권도현", "남유진", "석민재", "천보람", "표현우")

APPROVER_NAMES = ("구자현", "명수린", "봉태식", "설유나", "탁현빈")


In [ ]:
"""Mock MES 접속 클라이언트.

REST(/api)와 MCP(/mcp) 두 채널을 하나의 MesSnapshot으로 모은다.
표준 라이브러리만 사용한다. Fabric 노트북에 그대로 인라인되기 때문에
추가 패키지 설치가 있어선 안 된다.
"""

from __future__ import annotations

import datetime as dt
import json
import urllib.error
import urllib.request
from dataclasses import asdict, dataclass, field
from typing import Any, TypeVar

MES_BASE_URL = "https://mock-mes.greenrock-bb44c93a.koreacentral.azurecontainerapps.io"

# not_after 는 datetime 과 date 양쪽에 쓰인다. 둘을 섞어 넘기면 비교가 터지므로
# 같은 타입끼리만 묶이도록 제약한다.
_T = TypeVar("_T", dt.datetime, dt.date)


def parse_mcp_body(raw: str) -> dict | None:
    """MCP 응답 본문을 파싱한다.

    이 서버는 Accept 헤더에 text/event-stream이 있으면 SSE로 답한다.
    본문은 'event: message'로 시작하므로 'data:' 접두 검사만으로는
    SSE를 인식할 수 없다. 순수 JSON을 먼저 시도하고 실패하면 data 행을 모은다.
    """
    body = raw.strip()
    if not body:
        return None
    try:
        return json.loads(body)
    except json.JSONDecodeError:
        pass
    chunks = [
        line[len("data:"):].strip()
        for line in body.splitlines()
        if line.startswith("data:")
    ]
    if not chunks:
        raise ValueError(f"MCP 응답을 해석할 수 없습니다: {body[:200]!r}")
    return json.loads("".join(chunks))


def derive_equipment(process_results: list[dict], route: list[dict]) -> list[dict]:
    """공정이력의 eqp_id 고유값에서 설비 목록을 만든다.

    MES는 설비 마스터를 REST에도 MCP에도 노출하지 않는다. /equipment 웹 페이지에만
    있지만 HTML 파싱은 페이지 구조 변경에 취약하므로 쓰지 않는다.
    PKG 단계에 도달한 로트가 없어 EQP-PKG01은 여기서 빠진다.
    """
    eqp_type_by_step = {s["step_code"]: s.get("eqp_type") for s in route}
    first_step: dict[str, str] = {}
    for row in process_results:
        eqp_id = row.get("eqp_id")
        if eqp_id and eqp_id not in first_step:
            first_step[eqp_id] = row["step_code"]
    return [
        {
            "eqp_id": eqp_id,
            "eqp_type": eqp_type_by_step.get(first_step[eqp_id]),
            "step_code": first_step[eqp_id],
        }
        for eqp_id in sorted(first_step)
    ]


_SNAPSHOT_FIELDS = (
    "products",
    "materials",
    "bom",
    "lots",
    "process_results",
    "route",
    "equipment",
)


@dataclass(frozen=True)
class MesSnapshot:
    """QMS 생성기 전체의 유일한 입력. 네트워크 계층과 생성 계층의 경계다."""

    products: list[dict] = field(default_factory=list)
    materials: list[dict] = field(default_factory=list)
    bom: list[dict] = field(default_factory=list)
    lots: list[dict] = field(default_factory=list)
    process_results: list[dict] = field(default_factory=list)
    route: list[dict] = field(default_factory=list)
    equipment: list[dict] = field(default_factory=list)

    def to_dict(self) -> dict[str, list[dict]]:
        return asdict(self)

    @classmethod
    def from_dict(cls, payload: dict) -> "MesSnapshot":
        return cls(**{name: payload[name] for name in _SNAPSHOT_FIELDS})


def parse_mes_time(value: str) -> dt.datetime:
    """MES 시각 문자열을 tz-aware UTC 로 읽는다.

    한 컬럼에 aware 와 naive 가 섞이면 PySpark 가 둘을 다르게 저장한다.
    aware 는 calendar.timegm 을, naive 는 time.mktime(로컬 타임존)을 타므로
    드라이버가 UTC 가 아닌 곳에서는 같은 컬럼의 일부만 밀린다. 적재는 성공하고
    값만 틀리기 때문에 발견이 늦다. 그래서 입구에서 한 번에 통일한다.
    """
    parsed = dt.datetime.fromisoformat(value)
    if parsed.tzinfo is None:
        return parsed.replace(tzinfo=dt.timezone.utc)
    return parsed.astimezone(dt.timezone.utc)


def mes_anchor(snapshot: MesSnapshot) -> dt.datetime:
    """MES 공정이력의 마지막 종료 시각. QMS 데이터의 "지금"이다.

    두 가지 역할을 겸한다.

    첫째, 모든 QMS 시각의 기준점이다. QMS 시각을 벽시계 상수로 두면 MES 를
    재배포할 때 MES 만 움직이고 QMS 는 제자리에 남는다. MES 앵커가 Bicep 의
    utcNow() 로 배포 시점에 평가되므로 이 어긋남은 재배포마다 반드시 일어난다.

    둘째, 완료된 사건의 상한이다. 앵커는 곧 배포 시각이라 실습 시점의 "현재"와
    같다. MES 는 미래 데이터를 만들지 않으므로, QMS 만 앵커를 넘으면 아직 오지
    않은 날짜에 판정이 끝난 검사나 종결된 부적합이 생긴다.
    """
    if not snapshot.process_results:
        raise ValueError("공정이력이 비어 있어 QMS 시각의 기준점을 정할 수 없습니다.")
    return max(parse_mes_time(row["out_time"]) for row in snapshot.process_results)


def anchor_date(snapshot: MesSnapshot) -> dt.date:
    """앵커의 UTC 날짜. 날짜 컬럼들의 기준일이다."""
    return mes_anchor(snapshot).date()


def window_start(snapshot: MesSnapshot) -> dt.datetime:
    """MES 공정이력의 첫 종료 시각. 생산 구간의 시작이다."""
    if not snapshot.process_results:
        raise ValueError("공정이력이 비어 있어 생산 구간을 정할 수 없습니다.")
    return min(parse_mes_time(row["out_time"]) for row in snapshot.process_results)


def not_after(moment: _T, limit: _T) -> _T:
    """이미 일어난 사건의 시각을 현재(앵커) 이하로 자른다.

    조치 기한이나 유효성 점검 예정일처럼 아직 오지 않은 일은 미래가 정상이므로
    이 함수를 거치지 않는다. 완료를 뜻하는 값에만 쓴다.
    """
    return min(moment, limit)


class MesApiKeyMissing(RuntimeError):
    """API 키가 조달되지 않았을 때. 노트북 게이트 셀이 이 예외를 잡아 안내한다."""


class MesClient:
    def __init__(self, base_url: str = MES_BASE_URL, api_key: str = "", timeout: int = 60):
        if not api_key:
            raise MesApiKeyMissing(
                "MES_API_KEY가 비어 있습니다. 노트북 파라미터 셀에 키를 넣으세요."
            )
        self.base_url = base_url.rstrip("/")
        self.api_key = api_key
        self.timeout = timeout
        self._rpc_id = 0
        self._initialized = False

    def _open(self, url: str, data: bytes | None, headers: dict) -> str:
        request = urllib.request.Request(
            url, data=data, headers=headers, method="POST" if data else "GET"
        )
        with urllib.request.urlopen(request, timeout=self.timeout) as response:
            return response.read().decode("utf-8")

    def rest(self, path: str) -> Any:
        """REST GET. path 예: '/api/products'"""
        raw = self._open(
            self.base_url + path,
            None,
            {"X-API-Key": self.api_key, "Accept": "application/json"},
        )
        return json.loads(raw)

    def _rpc(self, method: str, params: dict, notify: bool = False) -> dict | None:
        payload: dict[str, Any] = {"jsonrpc": "2.0", "method": method, "params": params}
        if not notify:
            self._rpc_id += 1
            payload["id"] = self._rpc_id
        raw = self._open(
            self.base_url + "/mcp",
            json.dumps(payload).encode("utf-8"),
            {
                "Content-Type": "application/json",
                "Accept": "application/json, text/event-stream",
                "X-API-Key": self.api_key,
            },
        )
        return parse_mcp_body(raw)

    def _handshake(self) -> None:
        if self._initialized:
            return
        self._rpc(
            "initialize",
            {
                "protocolVersion": "2024-11-05",
                "capabilities": {},
                "clientInfo": {"name": "qms-seeder", "version": "1.0"},
            },
        )
        self._rpc("notifications/initialized", {}, notify=True)
        self._initialized = True

    def mcp_call(self, tool: str, args: dict | None = None) -> Any:
        """읽기 전용 MCP 툴 호출. 결과 리스트를 그대로 돌려준다."""
        if tool in {"start_lot", "register_process_result"}:
            raise ValueError(f"쓰기 툴 호출 금지: {tool}")
        self._handshake()
        response = self._rpc("tools/call", {"name": tool, "arguments": args or {}})
        if response is None:
            raise RuntimeError(f"MCP 툴 {tool} 응답이 비어 있습니다.")
        if "error" in response:
            raise RuntimeError(f"MCP 툴 {tool} 오류: {response['error']}")
        if "result" not in response:
            raise RuntimeError(f"MCP 툴 {tool} 응답에 result가 없습니다: {response}")
        result = response["result"]
        if result.get("isError"):
            raise RuntimeError(f"MCP 툴 {tool} 실패: {result.get('content')}")
        structured = result.get("structuredContent")
        if not isinstance(structured, dict) or "result" not in structured:
            raise RuntimeError(f"MCP 툴 {tool} 응답 형식이 예상과 다릅니다: {result}")
        return structured["result"]

    def fetch_snapshot(self) -> MesSnapshot:
        process_results = self.mcp_call("list_process_results", {"limit": 500})
        route = self.mcp_call("get_process_route")
        return MesSnapshot(
            products=self.rest("/api/products"),
            materials=self.rest("/api/materials"),
            bom=self.rest("/api/bom"),
            lots=self.mcp_call("list_lots", {"limit": 500}),
            process_results=process_results,
            route=route,
            equipment=derive_equipment(process_results, route),
        )


In [ ]:
"""QMS 마스터 3종. 불량코드 24 / 검사기준 108 / 검사원 15."""

from __future__ import annotations

import datetime as dt
import random


_SHIFTS = ("A", "B", "C")

# 검사원 자격 갱신 주기. 3년마다 재인증한다.
_CERTIFICATION_PERIOD_DAYS = 365 * 3


def build_defect_codes() -> list[dict]:
    """MES 상위 불량코드 6종을 QMS 세부코드 24종으로 전개한다."""
    meta = {code: (prefix, category, steps) for code, prefix, category, steps in DEFECT_TAXONOMY}
    rows = []
    for mes_code, seq, name_ko, name_en, severity, cause, action in DEFECT_DETAILS:
        prefix, category, steps = meta[mes_code]
        rows.append(
            {
                "defect_code": f"DEF-{prefix}-{seq:03d}",
                "defect_name_ko": name_ko,
                "defect_name_en": name_en,
                "mes_defect_code": mes_code,
                "defect_category": category,
                "severity": severity,
                "severity_score": SEVERITY_SCORE[severity],
                "typical_step_codes": steps,
                "standard_cause_ko": cause,
                "standard_action_ko": action,
                "is_active": True,
            }
        )
    return rows


def defect_codes_by_mes(defect_codes: list[dict]) -> dict[str, list[dict]]:
    grouped: dict[str, list[dict]] = {}
    for row in defect_codes:
        grouped.setdefault(row["mes_defect_code"], []).append(row)
    return grouped


def build_inspection_specs(snapshot: MesSnapshot) -> list[dict]:
    """제품 4 × 공정 9 × 특성 3 = 108행. 규격은 제품 노드에 따라 조정된다."""
    base_date = anchor_date(snapshot)
    product_names = {p["product_code"]: p["product_name"] for p in snapshot.products}
    step_names = {s["step_code"]: s["step_name"] for s in snapshot.route}
    rows = []
    for product_code in sorted(product_names):
        scale = PRODUCT_CD_SCALE[product_code]
        for step in sorted(snapshot.route, key=lambda s: s["seq"]):
            step_code = step["step_code"]
            for char_code in STEP_CHARACTERISTICS[step_code]:
                name_ko, meas_type, unit, target, lsl, usl = CHARACTERISTIC_BASE[char_code]
                if char_code == "CD":
                    target, lsl, usl = target * scale, lsl * scale, usl * scale
                sampling, sample_size = SAMPLING_METHODS[meas_type]
                rows.append(
                    {
                        "spec_id": f"SPEC-{product_code}-{step_code}-{char_code}",
                        "product_code": product_code,
                        "product_name": product_names[product_code],
                        "step_code": step_code,
                        "step_name": step_names[step_code],
                        "characteristic_code": char_code,
                        "characteristic_name_ko": name_ko,
                        "measurement_type": meas_type,
                        "unit": unit,
                        "target_value": round(target, 3),
                        "lsl": round(lsl, 3),
                        "usl": round(usl, 3),
                        "cpk_target": 1.33,
                        "sampling_method": sampling,
                        "sample_size": sample_size,
                        "inspection_frequency": INSPECTION_FREQUENCIES[char_code],
                        "control_method_ko": CONTROL_METHODS[meas_type],
                        "spec_version": "v1.2",
                        "effective_from": base_date - dt.timedelta(days=180),
                        "is_active": True,
                    }
                )
    return rows


def spec_index(specs: list[dict]) -> dict[tuple[str, str, str], dict]:
    return {(s["product_code"], s["step_code"], s["characteristic_code"]): s for s in specs}


def _certified_until(certified_from: dt.date, base_date: dt.date) -> dt.date:
    """검사원 자격 만료일. 활동 중이면 항상 현재보다 뒤에 있다.

    자격은 3년마다 갱신한다. 취득일에 3년을 한 번만 더하면 6년 전에 자격을 딴
    사람은 3년 전에 만료된 상태가 된다. 그 상태로 검사 기록을 만들면 자격 없는
    검사원이 수행한 검사가 대량으로 생긴다. 실제 QMS 에서 이는 그 자체로 중대
    부적합이라 데이터가 앞뒤로 맞지 않는다.

    그래서 취득일부터 3년 주기를 반복해 현재를 지나는 첫 만료일을 쓴다. 앵커가
    어디로 이동해도 활동 중인 검사원의 자격은 유효하다.
    """
    elapsed = (base_date - certified_from).days
    periods = elapsed // _CERTIFICATION_PERIOD_DAYS + 1
    return certified_from + dt.timedelta(days=_CERTIFICATION_PERIOD_DAYS * periods)


def build_inspectors(snapshot: MesSnapshot) -> list[dict]:
    """팀 5 × 교대 3 = 15명."""
    base_date = anchor_date(snapshot)
    rng = random.Random(SEED_MASTERS)
    rows = []
    for team_no, (team_ko, certified) in enumerate(INSPECTOR_TEAMS):
        for shift_no, shift in enumerate(_SHIFTS):
            index = team_no * len(_SHIFTS) + shift_no
            years = rng.randint(1, 6)
            certified_from = base_date - dt.timedelta(days=365 * years + rng.randint(0, 300))
            rows.append(
                {
                    "inspector_id": f"QI-{index + 1:03d}",
                    "inspector_name": INSPECTOR_NAMES[index],
                    "team_ko": team_ko,
                    "shift_code": shift,
                    "qualification_level": QUALIFICATION_LEVELS[index % len(QUALIFICATION_LEVELS)],
                    "certified_characteristics": certified,
                    "certified_from": certified_from,
                    "certified_until": _certified_until(certified_from, base_date),
                    "is_active": True,
                }
            )
    return rows


In [ ]:
"""qms_inspection 207행.

구성은 스펙 6.5절 그대로다.
  IPQC     91  MES 공정이력 1:1
  IPQC-RT  40  결함 보유(35) ∪ non-Pass(7), 교집합 2
  OQC      24  Done 로트 6 × 4 배치
  PCS      36  제품 4 × 공정 9
  EQV      16  설비 8 × 2회
"""

from __future__ import annotations

import datetime as dt
import random


INSPECTION_COUNTS = {"IPQC": 91, "IPQC-RT": 40, "OQC": 24, "PCS": 36, "EQV": 16}

# 공정능력조사(PCS)와 설비적격성(EQV)은 주간 근무조 안에서 수행한다. 시작
# 시각은 검사 종류마다 다르고 여기서는 그 근무조가 몇 시간짜리인지만 정한다.
# IPQC·IPQC-RT 는 공정 종료를 따라가고, OQC 는 로트 완료를 따라가므로 근무조와
# 무관하다. 팹이 24시간 돌아가니 출하검사가 새벽에 잡히는 것도 정상이다.
_SHIFT_LENGTH_HOURS = 8

_TEAM_BY_TYPE = {
    "IPQC": "계측팀",
    "IPQC-RT": "계측팀",
    "OQC": "출하검사팀",
    "PCS": "품질보증팀",
    "EQV": "신뢰성팀",
}

_BASIS = {
    "합격": "전 항목 규격 내, 관리한계 이탈 없음",
    "조건부합격": "일부 항목 관리한계 근접, 후속 공정 모니터링 조건부 승인",
    "불합격": "규격 이탈 확인, 부적합 보고서 발행",
}


def mes_result_index(snapshot: MesSnapshot) -> dict[int, dict]:
    return {row["id"]: row for row in snapshot.process_results}


def _lot_completion(snapshot: MesSnapshot) -> dict[str, dt.datetime]:
    """로트별 마지막 공정 종료 시각. 출하검사의 출발점이다."""
    done: dict[str, dt.datetime] = {}
    for row in snapshot.process_results:
        lot_id = row.get("lot_id")
        if not lot_id:
            continue
        moment = parse_mes_time(row["out_time"])
        if lot_id not in done or moment > done[lot_id]:
            done[lot_id] = moment
    return done


def _after(start: dt.datetime, minutes: int, as_of: dt.datetime) -> dt.datetime:
    """start 로부터 minutes 뒤. 앵커를 넘으면 남은 시간 안으로 접는다.

    검사는 공정이 끝난 뒤에 하므로 시각이 앞으로 간다. 그런데 앵커 직전에 끝난
    공정은 그 뒤에 검사할 시간이 아직 없다. 자르지 않으면 판정이 채워진 검사가
    미래에 놓인다.

    처음에는 앵커로 잘랐다. 그랬더니 잘린 것들이 앵커 시각 하나에 그대로
    쌓였다 — IPQC 2건과 IPQC-RT 3건이 정확히 같은 초에 놓였고, 그것이
    데이터에 있던 시각 중복의 전부였다. 적재도 검증도 통과한다. "가장 최근
    검사" 를 물으면 세 건이 같은 순간에 끝난 것으로 나온다.

    자르는 대신 남은 시간으로 나머지를 구해 접는다. 이미 뽑아 둔 minutes 를
    다시 쓰므로 난수를 더 쓰지 않는다. 조건에 따라 난수를 더 쓰면 앵커가
    움직일 때 뒤따르는 값이 통째로 밀린다 — 근무조 시각 선택에서 실제로
    겪었고, 검사 판정까지 바뀌었다.
    """
    when = start + dt.timedelta(minutes=minutes)
    if when <= as_of:
        return when
    room = int((as_of - start).total_seconds() // 60)
    if room <= 0:
        return as_of
    return start + dt.timedelta(minutes=minutes % (room + 1))


def _within_window(
    rng: random.Random, window: tuple[dt.datetime, dt.datetime], shift_hour: int
) -> dt.datetime:
    """생산 구간 안, 주간 근무조 시간대의 한 시각.

    출하검사(OQC)·공정능력조사(PCS)·설비적격성(EQV)은 특정 공정이력에서
    파생되지 않고 정기적으로 수행한다. 그래도 생산 구간 밖에 두면 안 된다.
    앞으로 나가면 아직 오지 않은 날짜에 완료된 검사가 생기고, 뒤로 물러나면
    이번 생산과 무관한 기록이 된다.

    구간 안에서 근무조 시간대에 드는 시각만 후보로 모아 그중 하나를 고른다.
    범위를 벗어난 값을 양끝으로 자르는 방식을 쓰면 잘린 행들이 경계 시각
    하나에 그대로 쌓인다. 후보를 미리 거르면 그 뭉침이 생기지 않는다.

    난수는 후보가 몇 개든 항상 두 번만 쓴다. rng.choice 와 분기별 randint 를
    쓰면 소비 횟수가 후보 개수에 따라 달라져, 그 뒤에 뽑는 검사원과 측정값까지
    통째로 밀린다. 앵커가 정수 일수로 움직일 때는 시(hour)가 그대로라 후보
    집합도 같아서 드러나지 않는다. 한 시간만 옮기면 18종 106칸이 바뀌었고
    그중에는 검사 판정 2건도 있었다.
    """
    start, end = window
    span_hours = int((end - start).total_seconds() // 3600)
    candidates = [
        moment
        for offset in range(span_hours + 1)
        if (moment := start + dt.timedelta(hours=offset)) + dt.timedelta(minutes=59) <= end
        and shift_hour <= moment.hour < shift_hour + _SHIFT_LENGTH_HOURS
    ]
    position = rng.random()
    minute = rng.randint(0, 59)
    if not candidates:
        # 구간이 근무조 하나보다 짧은 경우. 시간대를 포기하고 구간 안에서 고른다.
        seconds = max(int((end - start).total_seconds()), 0)
        return start + dt.timedelta(seconds=int(position * (seconds + 1)))
    return candidates[int(position * len(candidates))] + dt.timedelta(minutes=minute)


def _bounded(rng: random.Random, low: int, high: int, cap: int) -> int:
    """low..high 범위 정수를 cap 이하로 자른다. 수량 정합 검증을 항상 통과시킨다."""
    high = min(high, cap)
    low = min(low, high)
    return rng.randint(low, high)


def _sampling(rng: random.Random, wafer_cap: int) -> tuple[int, int]:
    """(inspected_wafer_qty, sample_size). sample_size 는 웨이퍼 매수 × 3포인트."""
    wafers = min(rng.randint(3, 5), wafer_cap)
    return wafers, min(wafers * 3, wafer_cap)


class _IdGen:
    def __init__(self) -> None:
        self.n = 0

    def next(self) -> str:
        self.n += 1
        return f"INS-2026-{self.n:04d}"


def build_inspections(snapshot: MesSnapshot, inspectors: list[dict]) -> list[dict]:
    as_of = mes_anchor(snapshot)
    window = (window_start(snapshot), as_of)
    rng = random.Random(SEED_INSPECTION)
    ids = _IdGen()
    by_team: dict[str, list[dict]] = {}
    for person in inspectors:
        by_team.setdefault(person["team_ko"], []).append(person)

    lots = {lot["lot_id"]: lot for lot in snapshot.lots}
    results = sorted(snapshot.process_results, key=lambda r: r["id"])

    rows: list[dict] = []
    rows.extend(_build_ipqc(rng, ids, results, lots, by_team, as_of))
    rows.extend(_build_retest(rng, ids, results, lots, by_team, as_of))
    rows.extend(_build_oqc(rng, ids, snapshot, by_team, as_of))
    rows.extend(_build_pcs(rng, ids, snapshot, by_team, window))
    rows.extend(_build_eqv(rng, ids, snapshot, by_team, window))
    return rows


def _row(
    ids: _IdGen,
    *,
    inspection_type: str,
    lot_id,
    product_code,
    product_name,
    step_code,
    step_name,
    eqp_id,
    mes_id,
    inspector: dict,
    when: dt.datetime,
    wafers: int,
    sample_size: int,
    judgment: str,
    defect_found_qty: int,
    measurement_count: int,
    has_ncr: bool,
    remark: str,
) -> dict:
    return {
        "inspection_id": ids.next(),
        "inspection_type": inspection_type,
        "lot_id": lot_id,
        "product_code": product_code,
        "product_name": product_name,
        "step_code": step_code,
        "step_name": step_name,
        "eqp_id": eqp_id,
        "mes_process_result_id": mes_id,
        "inspector_id": inspector["inspector_id"],
        "inspection_datetime": when,
        "sample_size": sample_size,
        "inspected_wafer_qty": wafers,
        "judgment": judgment,
        "judgment_basis_ko": _BASIS[judgment],
        "defect_found_qty": defect_found_qty,
        "measurement_count": measurement_count,
        "has_nonconformance": has_ncr,
        "remark_ko": remark,
    }


def _build_ipqc(rng, ids, results, lots, by_team, as_of) -> list[dict]:
    """MES 91건 1:1. 판정 분포는 스펙 6.1절을 그대로 따른다."""
    fails = [r for r in results if r["result"] == "Fail"]
    reworks = [r for r in results if r["result"] == "Rework"]
    pass_def = [r for r in results if r["result"] == "Pass" and r.get("defect_code")]
    pass_clean = [r for r in results if r["result"] == "Pass" and not r.get("defect_code")]

    plan: dict[int, tuple[str, bool]] = {}
    for row in fails:
        plan[row["id"]] = ("불합격", True)

    shuffled = list(reworks)
    rng.shuffle(shuffled)
    plan[shuffled[0]["id"]] = ("불합격", True)
    # 두번째 재작업건은 MES 불량코드가 있을 때만 조건부합격으로 둔다. 코드가
    # 없는데 조건부합격+결함검출이 겹치면 장치③ 신호(코드 없음+결함검출+불합격
    # 아님)와 우연히 일치해버린다.
    second = shuffled[1]
    plan[second["id"]] = ("조건부합격", True) if second.get("defect_code") else ("불합격", True)

    shuffled = list(pass_def)
    rng.shuffle(shuffled)
    for row in shuffled[:10]:
        plan[row["id"]] = ("조건부합격", True)
    for row in shuffled[10:]:
        plan[row["id"]] = ("합격", False)

    # 장치①과 장치③은 같은 51건 풀에서 나온다. 슬라이스를 겹치지 않게 잘라
    # 서로소를 구조적으로 보장한다. 구분은 defect_found_qty 0 / >0 로 이뤄진다.
    shuffled = list(pass_clean)
    rng.shuffle(shuffled)
    device1 = {r["id"] for r in shuffled[:3]}
    device3 = {r["id"] for r in shuffled[3:7]}
    for row in shuffled[:3]:
        plan[row["id"]] = ("불합격", True)
    for row in shuffled[3:7]:
        plan[row["id"]] = ("조건부합격", True)
    for row in shuffled[7:]:
        plan[row["id"]] = ("합격", False)

    rows = []
    for mes in results:
        judgment, has_ncr = plan[mes["id"]]
        wafers, sample_size = _sampling(rng, mes["in_qty"])
        if mes["id"] in device1:
            found = 0
            remark = "생산 통과분에 대해 품질 홀드 적용. 계측 재현성 확인 필요"
        elif mes["id"] in device3:
            found = _bounded(rng, 1, 4, sample_size)
            remark = "설비 판정에는 없던 결함을 검사에서 검출"
        elif judgment == "합격":
            found = 0
            remark = "정상 공정검사"
        else:
            found = _bounded(rng, 2, 9, sample_size)
            remark = "공정검사 중 결함 검출"
        rows.append(
            _row(
                ids,
                inspection_type="IPQC",
                lot_id=mes["lot_id"],
                product_code=lots[mes["lot_id"]]["product_code"],
                product_name=lots[mes["lot_id"]]["product_name"],
                step_code=mes["step_code"],
                step_name=mes["step_name"],
                eqp_id=mes["eqp_id"],
                mes_id=mes["id"],
                inspector=rng.choice(by_team[_TEAM_BY_TYPE["IPQC"]]),
                when=_after(parse_mes_time(mes["out_time"]), rng.randint(10, 240), as_of),
                wafers=wafers,
                sample_size=sample_size,
                judgment=judgment,
                defect_found_qty=found,
                measurement_count=0,
                has_ncr=has_ncr,
                remark=remark,
            )
        )
    return rows


def _build_retest(rng, ids, results, lots, by_team, as_of) -> list[dict]:
    """결함 보유 35 ∪ non-Pass 7 = 40건 재검사. 측정치 3점을 남긴다."""
    targets = [r for r in results if r.get("defect_code") or r["result"] != "Pass"]
    clean_mes = [r for r in targets if not r.get("defect_code")]
    coded_mes = [r for r in targets if r.get("defect_code")]

    # MES 불량코드가 없는 재검사 건은 전부 불합격으로 둔다. 그래야 장치③
    # (MES 코드 없음 + 결함 검출 + 불합격 아님)과 절대 충돌하지 않는다.
    plan = {r["id"]: ("불합격", True) for r in clean_mes}
    shuffled = list(coded_mes)
    rng.shuffle(shuffled)
    remaining = 10 - len(plan)
    for row in shuffled[:remaining]:
        plan[row["id"]] = ("불합격", True)
    for row in shuffled[remaining:remaining + 12]:
        plan[row["id"]] = ("조건부합격", False)
    for row in shuffled[remaining + 12:]:
        plan[row["id"]] = ("합격", False)

    rows = []
    for mes in targets:
        judgment, has_ncr = plan[mes["id"]]
        wafers, sample_size = _sampling(rng, mes["in_qty"])
        found = 0 if judgment == "합격" else _bounded(rng, 1, 8, sample_size)
        rows.append(
            _row(
                ids,
                inspection_type="IPQC-RT",
                lot_id=mes["lot_id"],
                product_code=lots[mes["lot_id"]]["product_code"],
                product_name=lots[mes["lot_id"]]["product_name"],
                step_code=mes["step_code"],
                step_name=mes["step_name"],
                eqp_id=mes["eqp_id"],
                mes_id=mes["id"],
                inspector=rng.choice(by_team[_TEAM_BY_TYPE["IPQC-RT"]]),
                when=_after(parse_mes_time(mes["out_time"]), rng.randint(300, 720), as_of),
                wafers=wafers,
                sample_size=sample_size,
                judgment=judgment,
                defect_found_qty=found,
                measurement_count=3,
                has_ncr=has_ncr,
                remark="재검사 수행. 계측 3점 기록",
            )
        )
    return rows


def _build_oqc(rng, ids, snapshot, by_team, as_of) -> list[dict]:
    """Done 로트 6건을 출하 배치 4개로 나눠 24건.

    출하검사는 로트 생산이 끝난 뒤 4~24시간 안에 한다. 전역 날짜가 아니라
    그 로트의 마지막 공정 종료 시각에서 유도하므로 로트마다 시점이 다르다.
    아직 생산 중인 로트(Running/Hold)에는 출하검사가 없다. "출하검사 대기
    로트"가 자연히 생기고, 이것 자체가 교차 질의 소재가 된다.
    """
    done = sorted((l for l in snapshot.lots if l["status"] == "Done"), key=lambda l: l["lot_id"])
    completed = _lot_completion(snapshot)
    step_names = {s["step_code"]: s["step_name"] for s in snapshot.route}
    plan_slots = [(lot, batch) for lot in done for batch in range(1, 5)]
    missing = [lot["lot_id"] for lot, _ in plan_slots if lot["lot_id"] not in completed]
    if missing:
        raise ValueError(f"Done 로트인데 MES 공정이력이 없습니다: {sorted(set(missing))}")
    order = list(range(len(plan_slots)))
    rng.shuffle(order)
    judgments = {}
    for rank, slot in enumerate(order):
        if rank < 2:
            judgments[slot] = ("불합격", True)
        elif rank < 6:
            judgments[slot] = ("조건부합격", True)
        else:
            judgments[slot] = ("합격", False)

    rows = []
    for slot, (lot, batch) in enumerate(plan_slots):
        judgment, has_ncr = judgments[slot]
        wafers, sample_size = _sampling(rng, lot["wafer_qty"])
        found = 0 if judgment == "합격" else _bounded(rng, 2, 9, sample_size)
        rows.append(
            _row(
                ids,
                inspection_type="OQC",
                lot_id=lot["lot_id"],
                product_code=lot["product_code"],
                product_name=lot["product_name"],
                step_code=lot["current_step"],
                step_name=step_names[lot["current_step"]],
                eqp_id=None,
                mes_id=None,
                inspector=rng.choice(by_team[_TEAM_BY_TYPE["OQC"]]),
                when=_after(completed[lot["lot_id"]], rng.randint(240, 1440), as_of),
                wafers=wafers,
                sample_size=sample_size,
                judgment=judgment,
                defect_found_qty=found,
                measurement_count=0,
                has_ncr=has_ncr,
                remark=f"출하 배치 {batch}/4 검사",
            )
        )
    return rows


def _build_pcs(rng, ids, snapshot, by_team, window) -> list[dict]:
    """제품 4 × 공정 9 = 36건 정기 공정능력 조사. 로트에 매이지 않는다."""
    products = sorted(snapshot.products, key=lambda p: p["product_code"])
    steps = sorted(snapshot.route, key=lambda s: s["seq"])
    slots = [(p, s) for p in products for s in steps]
    order = list(range(len(slots)))
    rng.shuffle(order)
    flagged = set(order[:7])

    rows = []
    for slot, (product, step) in enumerate(slots):
        judgment, has_ncr = ("조건부합격", True) if slot in flagged else ("합격", False)
        rows.append(
            _row(
                ids,
                inspection_type="PCS",
                lot_id=None,
                product_code=product["product_code"],
                product_name=product["product_name"],
                step_code=step["step_code"],
                step_name=step["step_name"],
                eqp_id=None,
                mes_id=None,
                inspector=rng.choice(by_team[_TEAM_BY_TYPE["PCS"]]),
                when=_within_window(rng, window, 9),
                wafers=3,
                sample_size=9,
                judgment=judgment,
                defect_found_qty=0,
                measurement_count=3,
                has_ncr=has_ncr,
                remark="정기 공정능력 조사. Cpk 산출용 3점 계측"
                if not has_ncr
                else "정기 공정능력 조사에서 Cpk 목표 1.33 미달",
            )
        )
    return rows


def _build_eqv(rng, ids, snapshot, by_team, window) -> list[dict]:
    """설비 8대 × 2회 = 16건 설비 검증. 제품과 로트 모두 무관하다."""
    step_names = {s["step_code"]: s["step_name"] for s in snapshot.route}
    slots = [(e, run) for e in snapshot.equipment for run in (1, 2)]
    order = list(range(len(slots)))
    rng.shuffle(order)
    flagged = set(order[:2])

    rows = []
    for slot, (equipment, run) in enumerate(slots):
        judgment, has_ncr = ("불합격", True) if slot in flagged else ("합격", False)
        rows.append(
            _row(
                ids,
                inspection_type="EQV",
                lot_id=None,
                product_code=None,
                product_name=None,
                step_code=equipment["step_code"],
                step_name=step_names[equipment["step_code"]],
                eqp_id=equipment["eqp_id"],
                mes_id=None,
                inspector=rng.choice(by_team[_TEAM_BY_TYPE["EQV"]]),
                when=_within_window(rng, window, 7),
                wafers=2,
                sample_size=6,
                judgment=judgment,
                defect_found_qty=_bounded(rng, 1, 5, 6) if has_ncr else 0,
                measurement_count=2,
                has_ncr=has_ncr,
                remark=f"설비 정기 검증 {run}회차",
            )
        )
    return rows


In [ ]:
"""qms_measurement 260행.

IPQC-RT 40×3 = 120, PCS 36×3 = 108, EQV 16×2 = 32.
IPQC 기본과 OQC는 합부만 남기고 측정치를 기록하지 않는다.
"""

from __future__ import annotations

import datetime as dt
import random


# 설비 검증은 특정 제품을 위한 검사가 아니다. 규격이 있어야 값을 뽑을 수 있으므로
# 기준 제품 하나를 정해 그 규격으로 측정한다.
EQV_SPEC_PRODUCT = "DDR5"

# 공정능력 미달(PCS 조건부합격)은 산포를 키워 표현한다. Cpk 는 약 0.6이 된다.
_DEGRADED_SIGMA_FACTOR = 2.2


def nominal_sigma(spec: dict) -> float:
    """규격폭의 1/8. 공칭 Cpk 가 1.33이 되는 산포다."""
    return (spec["usl"] - spec["lsl"]) / 8


def characteristics_for(inspection: dict) -> tuple[str, ...]:
    """검사 유형별 측정 특성. 개수가 measurement_count 와 항상 일치한다."""
    count = inspection["measurement_count"]
    if count == 0:
        return ()
    return STEP_CHARACTERISTICS[inspection["step_code"]][:count]


def _spec_product(inspection: dict) -> str:
    return inspection["product_code"] or EQV_SPEC_PRODUCT


def _draw(rng: random.Random, spec: dict, sigma: float, force_out: bool) -> float:
    lsl, usl, target = spec["lsl"], spec["usl"], spec["target_value"]
    if force_out:
        margin = (usl - lsl) * rng.uniform(0.04, 0.12)
        # 계수형(파티클수)은 하한이 0이라 아래로 이탈시키면 물리적으로 말이 안 된다.
        if spec["measurement_type"] == "계수형" or rng.random() < 0.5:
            value = usl + margin
        else:
            value = lsl - margin
    else:
        value = rng.gauss(target, sigma)
    if spec["unit"] == "ea":
        return float(max(0, round(value)))
    return round(value, 4)


def build_measurements(
    inspections: list[dict], specs: list[dict], as_of: dt.datetime
) -> list[dict]:
    rng = random.Random(SEED_MEASUREMENT)
    index = spec_index(specs)
    rows: list[dict] = []
    seq = 0

    for inspection in inspections:
        chars = characteristics_for(inspection)
        if not chars:
            continue
        product_code = _spec_product(inspection)
        degraded = inspection["judgment"] == "조건부합격" and inspection["has_nonconformance"]
        # 불합격 검사는 반드시 규격 이탈점을 하나 이상 남긴다. 판정과 측정이
        # 어긋나면 에이전트가 모순된 답을 하게 된다.
        force_index = 0 if inspection["judgment"] == "불합격" else -1

        for position, char_code in enumerate(chars):
            spec = index[(product_code, inspection["step_code"], char_code)]
            sigma = nominal_sigma(spec) * (_DEGRADED_SIGMA_FACTOR if degraded else 1.0)
            value = _draw(rng, spec, sigma, force_out=position == force_index)
            outside = value < spec["lsl"] or value > spec["usl"]
            seq += 1
            rows.append(
                {
                    "measurement_id": f"MEA-2026-{seq:06d}",
                    "inspection_id": inspection["inspection_id"],
                    "spec_id": spec["spec_id"],
                    "lot_id": inspection["lot_id"],
                    "product_code": product_code,
                    "step_code": inspection["step_code"],
                    "characteristic_code": char_code,
                    "characteristic_name_ko": CHARACTERISTIC_BASE[char_code][0],
                    "sample_no": position % max(inspection["inspected_wafer_qty"], 1) + 1,
                    "site_no": position + 1,
                    "measured_value": value,
                    "unit": spec["unit"],
                    "target_value": spec["target_value"],
                    "lsl": spec["lsl"],
                    "usl": spec["usl"],
                    "deviation_pct": round(
                        (value - spec["target_value"]) / spec["target_value"] * 100, 3
                    ),
                    "is_out_of_spec": outside,
                    "judgment": "NG" if outside else "OK",
                    "metrology_eqp_id": METROLOGY_EQP[char_code],
                    "measured_by": inspection["inspector_id"],
                    # 계측은 검사 중에 이뤄지므로 검사 시각 뒤로 5분씩 밀린다.
                    # 앵커 직전 검사는 그만큼의 시간이 아직 없으므로 자른다.
                    "measured_at": not_after(
                        inspection["inspection_datetime"]
                        + dt.timedelta(minutes=5 * (position + 1)),
                        as_of,
                    ),
                }
            )
    return rows


In [ ]:
"""qms_incoming_inspection 200행.

MES는 자재를 알지만 그 자재의 입고 품질은 남기지 않는다. 공급업체, 성적서,
샘플링 판정은 전부 QMS 고유 사실이다.
"""

from __future__ import annotations

import datetime as dt
import random


IQC_JUDGMENT_COUNTS = {"불합격": 24, "특채": 16, "합격": 160}

SUPPLIERS_BY_CATEGORY = {
    "Raw Wafer": ("SUP-A01", "SUP-A02"),
    "Chemical": ("SUP-B01", "SUP-B02"),
    "Gas": ("SUP-C01", "SUP-C02"),
    "Metal": ("SUP-D01", "SUP-D02"),
    "Mask": ("SUP-A01", "SUP-D02"),
    "Package": ("SUP-B02", "SUP-D01"),
}

MATERIAL_DEFECT_MAP = {
    "Raw Wafer": ("Scratch", "Particle"),
    "Chemical": ("Contamination", "Particle"),
    "Gas": ("Particle", "Contamination"),
    "Metal": ("Contamination",),
    "Mask": ("CD-OOS", "Overlay"),
    "Package": ("Scratch", "Contamination"),
}

# uom 별 1회 입고 수량 범위. 병 단위 가스와 미터 단위 와이어는 자릿수가 다르다.
_RECEIPT_QTY = {
    "EA": (200, 2000),
    "L": (20, 200),
    "BTL": (10, 60),
    "SET": (1, 6),
    "M": (1000, 8000),
    "KG": (50, 400),
}

_REMARKS = {
    "합격": "규격 이내. 정상 입고 처리",
    "특채": "경미한 규격 이탈. 사용처 한정 조건으로 특채 승인",
    "불합격": "규격 이탈 확인. 격리 후 부적합 보고서 발행",
}


def build_incoming_inspections(
    snapshot: MesSnapshot, inspectors: list[dict], defect_codes: list[dict]
) -> list[dict]:
    base_date = anchor_date(snapshot)
    rng = random.Random(SEED_INCOMING)
    suppliers = dict(SUPPLIERS)
    receiving = [i for i in inspectors if i["team_ko"] == "입고검사팀"]
    by_mes_defect = defect_codes_by_mes(defect_codes)
    materials = sorted(snapshot.materials, key=lambda m: m["material_code"])

    total = sum(IQC_JUDGMENT_COUNTS.values())
    slots = list(range(total))
    rng.shuffle(slots)
    judgments: dict[int, str] = {}
    cursor = 0
    for judgment in ("불합격", "특채", "합격"):
        count = IQC_JUDGMENT_COUNTS[judgment]
        for slot in slots[cursor : cursor + count]:
            judgments[slot] = judgment
        cursor += count

    rows = []
    for slot in range(total):
        material = materials[slot % len(materials)]
        category = material["category"]
        judgment = judgments[slot]
        supplier_code = SUPPLIERS_BY_CATEGORY[category][slot % 2]
        low, high = _RECEIPT_QTY[material["uom"]]
        received_qty = float(rng.randint(low, high))
        sample_size = min(rng.randint(3, 20), int(received_qty))
        # 입고일은 앵커에서 2~32일 전. 최소 2일을 띄우는 이유는 입고검사가
        # 입고 후 0~2일에 이뤄지기 때문이다. 0일부터 잡으면 검사일이 앵커를
        # 넘어 아직 오지 않은 날짜에 판정이 끝난 입고검사가 생긴다.
        receipt_date = base_date - dt.timedelta(days=rng.randint(2, 32))
        coa_received = rng.random() >= 0.10
        if not coa_received:
            coa_conformance = "미제출"
        elif judgment != "합격" and rng.random() < 0.7:
            coa_conformance = "불일치"
        else:
            coa_conformance = "일치"
        if judgment == "합격":
            defect_code = None
        else:
            mes_defect = rng.choice(MATERIAL_DEFECT_MAP[category])
            defect_code = rng.choice(by_mes_defect[mes_defect])["defect_code"]
        rows.append(
            {
                "iqc_id": f"IQC-2026-{slot + 1:04d}",
                "material_code": material["material_code"],
                "material_name": material["material_name"],
                "supplier_code": supplier_code,
                "supplier_name_ko": suppliers[supplier_code],
                "supplier_lot_no": f"{supplier_code[-3:]}-{receipt_date:%y%m}-{slot + 1:04d}",
                "receipt_date": receipt_date,
                "received_qty": received_qty,
                "uom": material["uom"],
                "sample_size": sample_size,
                "inspection_items_ko": INSPECTION_ITEMS[category],
                "judgment": judgment,
                "defect_code": defect_code,
                "coa_received": coa_received,
                "coa_conformance": coa_conformance,
                "inspector_id": rng.choice(receiving)["inspector_id"],
                "inspection_date": receipt_date + dt.timedelta(days=rng.randint(0, 2)),
                "remark_ko": _REMARKS[judgment],
            }
        )
    return rows


In [ ]:
"""qms_nonconformance 95행과 qms_disposition 95행.

둘은 1:1이고 날짜가 사슬로 이어지므로 한 함수에서 함께 만든다.
불일치 장치 ②(Fail인데 특채) ④(불합격 자재가 로트에 투입) ⑤(재작업 실패 후 폐기)가
여기서 심긴다.
"""

from __future__ import annotations

import datetime as dt
import random


NCR_SOURCE_COUNTS = {"공정검사": 43, "출하검사": 6, "입고검사": 40, "고객제기": 6}
CUSTOMER_COMPLAINT_COUNT = 6

_SOURCE_BY_INSPECTION_TYPE = {
    "IPQC": "공정검사",
    "IPQC-RT": "공정검사",
    "PCS": "공정검사",
    "EQV": "공정검사",
    "OQC": "출하검사",
}

STEP_DEFECT_MAP = {
    "DIFF": ("Contamination", "Particle"),
    "PHOTO": ("CD-OOS", "Overlay"),
    "ETCH": ("Etch-Residue", "CD-OOS"),
    "IMPL": ("Contamination",),
    "CVD": ("Particle", "Etch-Residue"),
    "CMP": ("Scratch", "Particle"),
    "METRO": ("CD-OOS", "Overlay"),
    "TEST": ("Contamination",),
    "PKG": ("Scratch",),
}

# '측정'은 장치① 전용으로 예약한다. 다른 NCR이 같은 원인을 쓰면 이야기가 흐려진다.
_GENERAL_CAUSES = ("설비", "자재", "작업방법", "환경")

_ROOT_CAUSE_TEXT = {
    "설비": "설비 파라미터가 점진적으로 드리프트해 관리한계를 벗어남",
    "자재": "입고 자재 로트 간 편차가 공정 결과로 전이됨",
    "작업방법": "개정된 작업표준이 현장 레시피에 반영되지 않음",
    "환경": "클린룸 온습도 변동이 공정 안정성에 영향을 줌",
    "측정": "계측 재현성 저하로 실제 품질과 판정이 어긋남",
}

_IMMEDIATE_ACTION = {
    "설비": "해당 설비 가동 중지 후 파라미터 재설정 및 검증 런 수행",
    "자재": "동일 공급 로트 전량 격리 및 대체 로트 투입",
    "작업방법": "작업표준 최신본 재배포 및 교대조 교육 실시",
    "환경": "공조 설정 재조정 및 파티클 모니터링 강화",
    "측정": "계측기 재교정 및 Gage R&R 재평가",
}

_ESTIMATED_COST = {"Critical": (20_000_000, 80_000_000), "Major": (5_000_000, 20_000_000), "Minor": (500_000, 5_000_000)}
_SCRAP_UNIT_COST = (1_200_000, 3_500_000)

_COMPLAINT_TEXT = "고객 현장에서 반환된 제품의 분석 결과 품질 이슈 확인"


def build_nonconformances(
    snapshot: MesSnapshot,
    inspections: list[dict],
    incoming: list[dict],
    defect_codes: list[dict],
) -> tuple[list[dict], list[dict]]:
    base_date = anchor_date(snapshot)
    rng = random.Random(SEED_NCR)
    mes = mes_result_index(snapshot)
    defect_by_code = {d["defect_code"]: d for d in defect_codes}
    by_mes_defect: dict[str, list[dict]] = {}
    for row in defect_codes:
        by_mes_defect.setdefault(row["mes_defect_code"], []).append(row)
    lots = {l["lot_id"]: l for l in snapshot.lots}
    step_names = {s["step_code"]: s["step_name"] for s in snapshot.route}
    products = sorted(snapshot.products, key=lambda p: p["product_code"])
    iqc_by_id = {r["iqc_id"]: r for r in incoming}

    seeds = _collect_seeds(inspections, incoming)
    devices = _pick_devices(rng, seeds, inspections, mes, snapshot, iqc_by_id)

    ncrs: list[dict] = []
    dispositions: list[dict] = []
    inspection_by_id = {i["inspection_id"]: i for i in inspections}

    for position, seed in enumerate(seeds):
        ncr_id = f"NCR-2026-{position + 1:04d}"
        if seed[0] == "inspection":
            ncr = _ncr_from_inspection(
                rng, ncr_id, inspection_by_id[seed[1]], mes, lots, by_mes_defect, devices, base_date
            )
        elif seed[0] == "iqc":
            ncr = _ncr_from_iqc(
                rng, ncr_id, iqc_by_id[seed[1]], defect_by_code, lots, step_names, devices, base_date
            )
        else:
            ncr = _ncr_from_complaint(
                rng, ncr_id, products[position % len(products)], defect_codes, base_date
            )
        disposition = _disposition_for(
            rng, f"DSP-2026-{position + 1:04d}", ncr, devices, step_names, base_date
        )
        if ncr["status"] == "완료":
            closed = disposition["decision_date"] + dt.timedelta(days=rng.randint(0, 3))
            if closed <= base_date:
                ncr["closed_date"] = closed
            else:
                # 종결 예정일이 아직 오지 않았다. 그대로 두면 미래에 종결된
                # 부적합이 생긴다. 최근 발견된 건이 아직 조사 중인 것이
                # 현실이므로 상태를 낮춘다. closed_date 는 None 으로 남는다.
                ncr["status"] = "조사중"
        ncrs.append(ncr)
        dispositions.append(disposition)
    return ncrs, dispositions


def _collect_seeds(inspections: list[dict], incoming: list[dict]) -> list[tuple[str, object]]:
    seeds: list[tuple[str, object]] = [
        ("inspection", i["inspection_id"]) for i in inspections if i["has_nonconformance"]
    ]
    seeds += [("iqc", r["iqc_id"]) for r in incoming if r["judgment"] != "합격"]
    seeds += [("complaint", n) for n in range(CUSTOMER_COMPLAINT_COUNT)]
    return seeds


def _pick_devices(rng, seeds, inspections, mes, snapshot, iqc_by_id) -> dict:
    """장치 ②④⑤가 붙을 대상을 미리 정한다. 건수를 확정적으로 고정하기 위함이다."""
    inspection_by_id = {i["inspection_id"]: i for i in inspections}
    fails, reworks = [], []
    for kind, key in seeds:
        if kind != "inspection":
            continue
        inspection = inspection_by_id[key]
        if inspection["inspection_type"] != "IPQC":
            continue
        result = mes[inspection["mes_process_result_id"]]["result"]
        if result == "Fail":
            fails.append(key)
        elif result == "Rework":
            reworks.append(key)

    rng.shuffle(fails)
    device2 = set(fails[:2])
    device5 = set(reworks)
    assert len(device2) == 2 and len(device5) == 2

    # 장치④: 불합격/특채 자재 중 BOM으로 실제 로트까지 이어지는 것만 후보다.
    # RETICLE-5NM 은 BOM에 없고, 로트가 도달하지 못한 공정의 자재도 이어지지 않는다.
    reachable = _reachable_bom(snapshot)
    candidates = [
        key for kind, key in seeds
        if kind == "iqc" and reachable.get(iqc_by_id[key]["material_code"])
    ]
    assert len(candidates) >= 2, "장치④ 후보 자재가 부족합니다."
    rng.shuffle(candidates)
    device4 = {}
    for key in candidates[:2]:
        material = iqc_by_id[key]["material_code"]
        product_code, step_code, lot_ids = rng.choice(reachable[material])
        device4[key] = (product_code, step_code, rng.choice(lot_ids))
    return {"device2": device2, "device4": device4, "device5": device5}


def _reachable_bom(snapshot: MesSnapshot) -> dict[str, list[tuple[str, str, list[str]]]]:
    """자재 → [(제품, 공정, 그 공정을 실제로 지난 로트들)]."""
    product_of = {l["lot_id"]: l["product_code"] for l in snapshot.lots}
    lots_at: dict[tuple[str, str], list[str]] = {}
    for row in snapshot.process_results:
        key = (product_of[row["lot_id"]], row["step_code"])
        bucket = lots_at.setdefault(key, [])
        if row["lot_id"] not in bucket:
            bucket.append(row["lot_id"])
    reachable: dict[str, list[tuple[str, str, list[str]]]] = {}
    for entry in snapshot.bom:
        key = (entry["product_code"], entry["step_code"])
        if key in lots_at:
            reachable.setdefault(entry["material_code"], []).append(
                (entry["product_code"], entry["step_code"], sorted(lots_at[key]))
            )
    return reachable


def _base_ncr(rng, ncr_id: str, source: str, defect: dict, detected: dt.date) -> dict:
    cause = rng.choice(_GENERAL_CAUSES)
    low, high = _ESTIMATED_COST[defect["severity"]]
    status = rng.choices(
        ("완료", "조사중", "처리대기", "접수", "보류"), weights=(45, 20, 15, 15, 5)
    )[0]
    return {
        "ncr_id": ncr_id,
        "ncr_source": source,
        "inspection_id": None,
        "iqc_id": None,
        "lot_id": None,
        "product_code": None,
        "product_name": None,
        "step_code": None,
        "step_name": None,
        "material_code": None,
        "eqp_id": None,
        "defect_code": defect["defect_code"],
        "mes_defect_code": defect["mes_defect_code"],
        "severity": defect["severity"],
        "affected_qty": 1,
        "detected_date": detected,
        "root_cause_category": cause,
        "root_cause_ko": _ROOT_CAUSE_TEXT[cause],
        "immediate_action_ko": _IMMEDIATE_ACTION[cause],
        "owner_dept_ko": rng.choice(OWNER_DEPTS),
        "owner_name": rng.choice(OWNER_NAMES),
        "status": status,
        "due_date": detected + dt.timedelta(days=rng.randint(7, 14)),
        "closed_date": None,
        "estimated_cost_krw": rng.randrange(low, high, 100_000),
    }


def _ncr_from_inspection(rng, ncr_id, inspection, mes, lots, by_mes_defect, devices, as_of) -> dict:
    mes_row = mes.get(inspection["mes_process_result_id"]) if inspection["mes_process_result_id"] else None
    if mes_row and mes_row.get("defect_code"):
        parent = mes_row["defect_code"]
    else:
        parent = rng.choice(STEP_DEFECT_MAP[inspection["step_code"]])
    defect = rng.choice(by_mes_defect[parent])
    detected = not_after(
        inspection["inspection_datetime"].date() + dt.timedelta(days=rng.randint(0, 2)), as_of
    )
    ncr = _base_ncr(
        rng, ncr_id, _SOURCE_BY_INSPECTION_TYPE[inspection["inspection_type"]], defect, detected
    )
    ncr["inspection_id"] = inspection["inspection_id"]
    ncr["lot_id"] = inspection["lot_id"]
    ncr["product_code"] = inspection["product_code"]
    ncr["product_name"] = inspection["product_name"]
    ncr["step_code"] = inspection["step_code"]
    ncr["step_name"] = inspection["step_name"]
    ncr["eqp_id"] = inspection["eqp_id"]

    if mes_row is not None:
        ncr["affected_qty"] = max(1, min(rng.randint(1, 12), mes_row["out_qty"]))
        is_quality_hold = (
            mes_row["result"] == "Pass"
            and not mes_row.get("defect_code")
            and inspection["judgment"] == "불합격"
            and inspection["defect_found_qty"] == 0
        )
        if is_quality_hold:
            ncr["root_cause_category"] = "측정"
            ncr["root_cause_ko"] = _ROOT_CAUSE_TEXT["측정"]
            ncr["immediate_action_ko"] = _IMMEDIATE_ACTION["측정"]
    elif inspection["lot_id"]:
        ncr["affected_qty"] = max(1, min(rng.randint(1, 12), lots[inspection["lot_id"]]["wafer_qty"]))
    else:
        ncr["affected_qty"] = rng.randint(1, 8)
    return ncr


def _ncr_from_iqc(rng, ncr_id, iqc, defect_by_code, lots, step_names, devices, as_of) -> dict:
    defect = defect_by_code[iqc["defect_code"]]
    detected = not_after(iqc["inspection_date"] + dt.timedelta(days=rng.randint(0, 2)), as_of)
    ncr = _base_ncr(rng, ncr_id, "입고검사", defect, detected)
    ncr["iqc_id"] = iqc["iqc_id"]
    ncr["material_code"] = iqc["material_code"]
    ncr["affected_qty"] = max(1, min(rng.randint(1, 50), int(iqc["received_qty"])))
    ncr["root_cause_category"] = "자재"
    ncr["root_cause_ko"] = _ROOT_CAUSE_TEXT["자재"]
    ncr["immediate_action_ko"] = _IMMEDIATE_ACTION["자재"]

    traced = devices["device4"].get(iqc["iqc_id"])
    if traced:
        product_code, step_code, lot_id = traced
        ncr["product_code"] = product_code
        ncr["product_name"] = lots[lot_id]["product_name"]
        ncr["step_code"] = step_code
        ncr["step_name"] = step_names[step_code]
        ncr["lot_id"] = lot_id
        ncr["immediate_action_ko"] = "해당 자재가 투입된 로트를 역추적해 후속 공정 홀드"
    return ncr


def _ncr_from_complaint(rng, ncr_id, product, defect_codes, base_date) -> dict:
    defect = rng.choice(defect_codes)
    detected = base_date - dt.timedelta(days=rng.randint(3, 10))
    ncr = _base_ncr(rng, ncr_id, "고객제기", defect, detected)
    ncr["product_code"] = product["product_code"]
    ncr["product_name"] = product["product_name"]
    ncr["affected_qty"] = rng.randint(1, 10)
    ncr["root_cause_ko"] = _COMPLAINT_TEXT
    return ncr


def _disposition_for(
    rng, disposition_id: str, ncr: dict, devices: dict, step_names: dict, as_of: dt.date
) -> dict:
    inspection_id = ncr["inspection_id"]
    rework_step, rework_result, scrap_cost = None, None, 0

    if inspection_id in devices["device2"]:
        disposition_type, decision_body = "특채", "MRB"
    elif inspection_id in devices["device5"]:
        # 재작업을 시도했으나 실패해 결국 폐기된 건. 두 사실이 한 행에 함께 남는다.
        disposition_type, decision_body = "폐기", "MRB"
        rework_step, rework_result = ncr["step_code"], "실패"
    elif ncr["ncr_source"] == "입고검사":
        disposition_type = "반품" if ncr["defect_code"] and ncr["severity"] != "Minor" else "특채"
        decision_body = "품질책임자"
    elif ncr["ncr_source"] == "고객제기":
        disposition_type = rng.choice(("선별", "폐기", "반품"))
        decision_body = "품질책임자"
    else:
        # 특채는 장치②와 입고검사에만 허용한다. 여기서 새면 장치② 건수가 흔들린다.
        disposition_type = rng.choice(("재작업", "선별", "폐기"))
        decision_body = rng.choice(("MRB", "품질책임자", "생산책임자"))
        if disposition_type == "재작업":
            rework_step = ncr["step_code"]
            rework_result = rng.choice(("성공", "성공", "진행중"))

    disposition_qty = rng.randint(1, ncr["affected_qty"])
    if disposition_type == "폐기":
        scrap_cost = disposition_qty * rng.randrange(*_SCRAP_UNIT_COST, 100_000)
    decision_date = not_after(
        ncr["detected_date"] + dt.timedelta(days=rng.randint(1, 5)), as_of
    )
    return {
        "disposition_id": disposition_id,
        "ncr_id": ncr["ncr_id"],
        "lot_id": ncr["lot_id"],
        "product_code": ncr["product_code"],
        "step_code": ncr["step_code"],
        "disposition_type": disposition_type,
        "disposition_qty": disposition_qty,
        "decision_date": decision_date,
        "decision_body_ko": decision_body,
        "approver_name": rng.choice(APPROVER_NAMES),
        "approval_status": rng.choices(("승인", "대기", "반려"), weights=(85, 10, 5))[0],
        "rework_step_code": rework_step,
        "rework_result": rework_result,
        "scrap_cost_krw": scrap_cost,
        "effectiveness_check_date": decision_date + dt.timedelta(days=rng.randint(7, 21)),
        "effectiveness_result": rng.choices(("유효", "재발", "확인중"), weights=(70, 10, 20))[0],
        "reason_ko": f"{ncr['severity']} 등급 부적합에 대한 {disposition_type} 결정",
    }


In [ ]:
"""스펙 8절 검증 8항목.

장치 검출 질의는 데이터 값만으로 성립해야 한다. 전용 플래그 컬럼을 두면
검증은 쉬워지지만 '두 시스템을 함께 봐야 답이 나온다'는 전제가 무너진다.
"""

from __future__ import annotations

import datetime as dt
from dataclasses import dataclass


TABLE_ROW_TARGETS = {
    "qms_defect_code": 24,
    "qms_inspection_spec": 108,
    "qms_inspector": 15,
    "qms_inspection": 207,
    "qms_measurement": 260,
    "qms_incoming_inspection": 200,
    "qms_nonconformance": 95,
    "qms_disposition": 95,
}

FORBIDDEN_COLUMNS = frozenset({"mes_result", "scrap_qty", "operator", "in_qty", "out_qty"})

DEVICE_TARGETS = {
    "① MES Pass ↔ QMS 불합격": 3,
    "② MES Fail ↔ QMS 특채": 2,
    "③ MES 불량코드 없음 ↔ QMS 결함 검출": 4,
    "④ IQC 불합격 자재가 로트에 투입": 2,
    "⑤ MES Rework ↔ 재작업 실패 후 폐기": 2,
}


class ValidationError(RuntimeError):
    """치명 항목이 실패했을 때. 적재를 중단시킨다."""


@dataclass(frozen=True)
class ValidationResult:
    name: str
    passed: bool
    detail: str
    fatal: bool


def validate(snapshot: MesSnapshot, tables: dict[str, list[dict]]) -> list[ValidationResult]:
    return [
        _check_row_counts(tables),
        _check_orphan_keys(snapshot, tables),
        _check_forbidden_columns(tables),
        _check_internal_references(tables),
        _check_timestamp_timezones(tables),
        _check_time_column_coverage(tables),
        _check_time_causality(snapshot, tables),
        _check_no_future_completions(snapshot, tables),
        _check_inspector_certification(snapshot, tables),
        _check_quantities(tables),
        _check_measurement_limits(tables),
        _check_denormalized_specs(tables),
        _check_devices(snapshot, tables),
    ]


def raise_on_fatal(results: list[ValidationResult]) -> None:
    fatal = [r for r in results if r.fatal and not r.passed]
    if fatal:
        raise ValidationError("치명 검증 실패:\n" + "\n".join(f"- {r.name}: {r.detail}" for r in fatal))


def format_report(results: list[ValidationResult]) -> str:
    width = max(len(r.name) for r in results)
    lines = [f"{'항목'.ljust(width)}  결과  상세", "-" * (width + 40)]
    for result in results:
        lines.append(f"{result.name.ljust(width)}  {'PASS' if result.passed else 'FAIL'}  {result.detail}")
    return "\n".join(lines)


def _result(name: str, problems: list[str], ok_detail: str, fatal: bool = False) -> ValidationResult:
    if problems:
        return ValidationResult(name, False, "; ".join(problems[:5]), fatal)
    return ValidationResult(name, True, ok_detail, fatal)


def _check_row_counts(tables) -> ValidationResult:
    problems = [
        f"{name} {len(tables[name])}행 (목표 {target})"
        for name, target in TABLE_ROW_TARGETS.items()
        if len(tables[name]) != target
    ]
    return _result("행수", problems, f"8개 테이블 총 {sum(len(v) for v in tables.values())}행")


def _check_orphan_keys(snapshot, tables) -> ValidationResult:
    known = {
        "lot_id": {l["lot_id"] for l in snapshot.lots},
        "product_code": {p["product_code"] for p in snapshot.products},
        "step_code": {s["step_code"] for s in snapshot.route},
        "material_code": {m["material_code"] for m in snapshot.materials},
        "eqp_id": {e["eqp_id"] for e in snapshot.equipment},
        "mes_process_result_id": {r["id"] for r in snapshot.process_results},
    }
    problems = []
    for table_name, rows in tables.items():
        for row in rows:
            for column, allowed in known.items():
                value = row.get(column)
                if value is not None and value not in allowed:
                    problems.append(f"{table_name}.{column}={value}")
    return _result("고아 키", problems, "MES 미존재 키 0건", fatal=True)


def _check_forbidden_columns(tables) -> ValidationResult:
    problems = []
    for table_name, rows in tables.items():
        present = set().union(*(row.keys() for row in rows)) if rows else set()
        for column in sorted(present & FORBIDDEN_COLUMNS):
            problems.append(f"{table_name}.{column}")
    return _result("무중복 위반", problems, "MES 중복 컬럼 0건", fatal=True)


def _check_internal_references(tables) -> ValidationResult:
    inspection_ids = {r["inspection_id"] for r in tables["qms_inspection"]}
    spec_ids = {r["spec_id"] for r in tables["qms_inspection_spec"]}
    inspector_ids = {r["inspector_id"] for r in tables["qms_inspector"]}
    defect_ids = {r["defect_code"] for r in tables["qms_defect_code"]}
    ncr_ids = {r["ncr_id"] for r in tables["qms_nonconformance"]}
    iqc_ids = {r["iqc_id"] for r in tables["qms_incoming_inspection"]}

    references = [
        ("qms_inspection.inspector_id", [r["inspector_id"] for r in tables["qms_inspection"]], inspector_ids),
        ("qms_measurement.inspection_id", [r["inspection_id"] for r in tables["qms_measurement"]], inspection_ids),
        ("qms_measurement.spec_id", [r["spec_id"] for r in tables["qms_measurement"]], spec_ids),
        ("qms_measurement.measured_by", [r["measured_by"] for r in tables["qms_measurement"]], inspector_ids),
        ("qms_incoming_inspection.inspector_id", [r["inspector_id"] for r in tables["qms_incoming_inspection"]], inspector_ids),
        ("qms_incoming_inspection.defect_code", [r["defect_code"] for r in tables["qms_incoming_inspection"]], defect_ids),
        ("qms_nonconformance.inspection_id", [r["inspection_id"] for r in tables["qms_nonconformance"]], inspection_ids),
        ("qms_nonconformance.iqc_id", [r["iqc_id"] for r in tables["qms_nonconformance"]], iqc_ids),
        ("qms_nonconformance.defect_code", [r["defect_code"] for r in tables["qms_nonconformance"]], defect_ids),
        ("qms_disposition.ncr_id", [r["ncr_id"] for r in tables["qms_disposition"]], ncr_ids),
    ]
    problems = []
    for label, values, allowed in references:
        missing = {v for v in values if v is not None and v not in allowed}
        if missing:
            problems.append(f"{label} → {sorted(missing)[:3]}")
    return _result("내부 FK", problems, "내부 참조 전건 유효", fatal=True)


def _check_timestamp_timezones(tables) -> ValidationResult:
    """모든 시각 값이 tz-aware 인지 본다.

    PySpark 는 tz-aware 면 calendar.timegm 을, naive 면 time.mktime(로컬 타임존)을
    탄다. 한 컬럼에 둘이 섞이면 드라이버가 UTC 가 아닌 곳에서 일부 행만 밀리는데,
    적재는 성공하고 값만 틀리므로 눈으로는 찾을 수 없다. 그래서 치명 항목이다.
    dt.date 는 dt.datetime 의 인스턴스가 아니므로 날짜 컬럼은 걸리지 않는다.
    """
    naive: dict[str, int] = {}
    for name, rows in tables.items():
        for row in rows:
            for column, value in row.items():
                if isinstance(value, dt.datetime) and value.tzinfo is None:
                    key = f"{name}.{column}"
                    naive[key] = naive.get(key, 0) + 1
    problems = [f"{key} naive {count}건" for key, count in sorted(naive.items())]
    return _result("타임존 통일", problems, "모든 시각이 tz-aware UTC", fatal=True)


def _check_time_causality(snapshot, tables) -> ValidationResult:
    out_time = {r["id"]: parse_mes_time(r["out_time"]) for r in snapshot.process_results}
    inspection_by_id = {r["inspection_id"]: r for r in tables["qms_inspection"]}
    ncr_by_id = {r["ncr_id"]: r for r in tables["qms_nonconformance"]}
    problems = []

    # QMS 시각이 MES 구간에서 떨어져 나가지 않았는지 본다. 벽시계 상수가 다시
    # 들어오면 MES 재배포 때 여기부터 어긋나므로, 창을 앵커 기준으로 잡는다.
    window_start = min(out_time.values())
    window_end = max(out_time.values()) + dt.timedelta(days=4)

    for row in tables["qms_inspection"]:
        mes_id = row["mes_process_result_id"]
        mes_out = out_time.get(mes_id) if mes_id is not None else None
        when = row["inspection_datetime"]
        if mes_out is not None and when < mes_out:
            problems.append(f"{row['inspection_id']} 검사시각이 MES 종료시각보다 이르다")
        if not window_start <= when <= window_end:
            problems.append(f"{row['inspection_id']} 검사시각이 MES 구간 밖이다 ({when})")
    for row in tables["qms_nonconformance"]:
        inspection = inspection_by_id.get(row["inspection_id"])
        if inspection and row["detected_date"] < inspection["inspection_datetime"].date():
            problems.append(f"{row['ncr_id']} 검출일이 검사일보다 이르다")
    for row in tables["qms_disposition"]:
        if row["decision_date"] < ncr_by_id[row["ncr_id"]]["detected_date"]:
            problems.append(f"{row['disposition_id']} 결정일이 검출일 이전이다")
    return _result("시간 인과", problems, "검사 → 부적합 → 처리 순서 성립")


# 완료를 뜻하는 컬럼. 앵커(데이터의 현재)를 넘으면 아직 오지 않은 날짜에
# 끝난 사건이 된다.
_COMPLETED_COLUMNS = [
    ("qms_inspection_spec", "effective_from"),
    ("qms_inspector", "certified_from"),
    ("qms_inspection", "inspection_datetime"),
    ("qms_measurement", "measured_at"),
    ("qms_incoming_inspection", "receipt_date"),
    ("qms_incoming_inspection", "inspection_date"),
    ("qms_nonconformance", "detected_date"),
    ("qms_nonconformance", "closed_date"),
    ("qms_disposition", "decision_date"),
]

# 아직 오지 않은 일. 미래에 있는 것이 정상이라 상한을 걸지 않는다.
# 여기까지 과거로 끌어내리면 "기한이 임박한 미결 부적합" 같은 질문이 죽는다.
_PLANNED_COLUMNS = [
    ("qms_inspector", "certified_until"),
    ("qms_nonconformance", "due_date"),
    ("qms_disposition", "effectiveness_check_date"),
]


def _declared_time_columns() -> set[tuple[str, str]]:
    """DDL 이 선언한 DATE·TIMESTAMP 컬럼 전부.

    분류 목록을 손으로 적으면 새 컬럼이 조용히 빠진다. 컬럼 이름에 date 나
    time 이 들어가는지로 거르는 방식도 같은 함정이다. measured_at 은 둘 다
    없어서 그런 필터에 걸리지 않는다. 그래서 선언된 타입에서 뽑는다.
    """
    columns = set()
    for table, ddl in TABLE_DDL.items():
        for field in ddl.split(","):
            parts = field.strip().rsplit(" ", 1)
            if len(parts) == 2 and parts[1] in ("DATE", "TIMESTAMP"):
                columns.add((table, parts[0]))
    return columns


def _check_time_column_coverage(tables) -> ValidationResult:
    """모든 시각 컬럼이 완료·예정 중 하나로 분류됐는지.

    분류에서 빠진 컬럼은 미래 검사를 그냥 통과한다. 검증이 늘 PASS 라
    안전해 보이지만 실제로는 그 컬럼을 아무도 보고 있지 않다.
    """
    classified = {c for c in _COMPLETED_COLUMNS} | {c for c in _PLANNED_COLUMNS}
    declared = _declared_time_columns()
    problems = [
        f"{table}.{column} 이 완료·예정 어느 쪽으로도 분류되지 않았다"
        for table, column in sorted(declared - classified)
    ]
    problems += [
        f"{table}.{column} 은 DDL 에 없는 컬럼이다"
        for table, column in sorted(classified - declared)
    ]
    return _result(
        "시각 컬럼 분류", problems, f"DATE·TIMESTAMP {len(declared)}개 전부 분류됨", fatal=True
    )


def _check_no_future_completions(snapshot, tables) -> ValidationResult:
    """이미 끝난 사건이 현재를 넘지 않는지.

    앵커는 MES 배포 시각이라 실습 시점의 "지금"과 같다. 판정이 채워진 검사나
    종결된 부적합이 앵커를 넘으면, 참가자가 "최근 검사 결과"를 물었을 때 아직
    오지 않은 날짜의 합격 판정이 돌아온다. 적재는 성공하므로 데이터를 직접
    들여다보기 전에는 드러나지 않는다.
    """
    anchor = mes_anchor(snapshot)
    limit = anchor.date()
    problems = []
    for table, column in _COMPLETED_COLUMNS:
        for row in tables[table]:
            value = row.get(column)
            if value is None:
                continue
            # date 와 datetime 은 서로 비교할 수 없다. 컬럼 타입에 맞춰 자른다.
            ceiling = anchor if isinstance(value, dt.datetime) else limit
            if value > ceiling:
                problems.append(f"{table}.{column} {value} 가 현재({ceiling})를 넘는다")
    return _result(
        "미래 완료 사건",
        problems,
        f"완료 컬럼 {len(_COMPLETED_COLUMNS)}종 전부 앵커({anchor:%Y-%m-%d %H:%M}) 이하",
        fatal=True,
    )


def _check_inspector_certification(snapshot, tables) -> ValidationResult:
    """검사를 수행한 사람의 자격이 그 시점에 유효했는지.

    자격이 만료된 검사원의 기록은 실제 QMS 에서 그 자체로 중대 부적합이다.
    이 데이터에서는 의도한 장치가 아니므로 한 건도 없어야 한다.
    """
    inspectors = {r["inspector_id"]: r for r in tables["qms_inspector"]}
    problems = []
    for row in tables["qms_inspection"]:
        person = inspectors.get(row["inspector_id"])
        if person is None:
            continue
        when = row["inspection_datetime"].date()
        if not (person["certified_from"] <= when <= person["certified_until"]):
            problems.append(
                f"{row['inspection_id']} 를 자격 범위 밖의 {row['inspector_id']} 가 수행했다"
            )
    detail = f"검사 {len(tables['qms_inspection'])}건 전부 유효 자격자가 수행"
    return _result("검사원 자격", problems, detail, fatal=True)


def _check_quantities(tables) -> ValidationResult:
    affected = {r["ncr_id"]: r["affected_qty"] for r in tables["qms_nonconformance"]}
    problems = []
    for row in tables["qms_inspection"]:
        if row["defect_found_qty"] > row["sample_size"]:
            problems.append(f"{row['inspection_id']} 검출수 > 샘플수")
    for row in tables["qms_disposition"]:
        if row["disposition_qty"] > affected[row["ncr_id"]]:
            problems.append(f"{row['disposition_id']} 처리수량 > 영향수량")
    return _result("수량 정합", problems, "수량 대소관계 성립")


def _check_measurement_limits(tables) -> ValidationResult:
    problems = []
    for row in tables["qms_measurement"]:
        outside = row["measured_value"] < row["lsl"] or row["measured_value"] > row["usl"]
        if outside != row["is_out_of_spec"]:
            problems.append(f"{row['measurement_id']} 규격이탈 플래그 불일치")
        if row["judgment"] != ("NG" if outside else "OK"):
            problems.append(f"{row['measurement_id']} 판정 불일치")
    return _result("측정치 규격", problems, "규격 이탈 플래그 전건 일치")


_DENORMALIZED_SPEC_FIELDS = ("unit", "target_value", "lsl", "usl")


def _check_denormalized_specs(tables) -> ValidationResult:
    """계측이 들고 있는 규격이 그 spec_id 의 것과 같은지 본다.

    qms_measurement 는 규격 네 값을 자기 행에 복제해 둔다. 덕분에 규격 이탈
    판정에 조인이 필요 없고, 참가자가 characteristic_code 로 잘못 조인할 이유도
    없어진다. 그 복제가 어긋나면 문서가 "조인하지 마세요"라고 안내하는 근거가
    사라지므로 치명으로 둔다.

    characteristic_code 는 여섯 종뿐이라 스펙 108행에서 유일하지 않다. CD 하나에
    32행이 걸리고 제품마다 목표가 4배까지 다르다. 그 조인은 22.65배로 늘어나며
    짝의 95.6%가 다른 제품의 규격이다.
    """
    specs = {s["spec_id"]: s for s in tables["qms_inspection_spec"]}
    drift = []
    for row in tables["qms_measurement"]:
        spec = specs.get(row["spec_id"])
        if spec is None:
            drift.append(f"{row['measurement_id']}: spec_id 없음")
            continue
        for field in _DENORMALIZED_SPEC_FIELDS:
            if row[field] != spec[field]:
                drift.append(f"{row['measurement_id']}.{field} {row[field]} != {spec[field]}")
    return ValidationResult(
        "계측 규격 복제",
        not drift,
        f"규격 4값 복제 {len(tables['qms_measurement'])}건 전부 일치"
        if not drift
        else f"복제 불일치 {len(drift)}건: {drift[:3]}",
        fatal=True,
    )


def _check_devices(snapshot, tables) -> ValidationResult:
    counts = count_devices(snapshot, tables)
    problems = [
        f"{name} {counts[name]}건 (설계 {target}건)"
        for name, target in DEVICE_TARGETS.items()
        if counts[name] != target
    ]
    return _result("불일치 장치", problems, "5종 13건 전부 검출")


def count_devices(snapshot: MesSnapshot, tables: dict[str, list[dict]]) -> dict[str, int]:
    """전용 표시 컬럼 없이 값 조건만으로 장치를 세어 본다."""
    mes = {r["id"]: r for r in snapshot.process_results}
    inspection_by_id = {r["inspection_id"]: r for r in tables["qms_inspection"]}
    disposition_by_ncr = {r["ncr_id"]: r for r in tables["qms_disposition"]}
    lots = {l["lot_id"] for l in snapshot.lots}

    device1 = device3 = device2 = device4 = device5 = 0

    for row in tables["qms_inspection"]:
        mes_id = row["mes_process_result_id"]
        if mes_id is None:
            continue
        source = mes.get(mes_id)
        if source is None or source.get("defect_code"):
            continue
        if source["result"] == "Pass" and row["judgment"] == "불합격" and row["defect_found_qty"] == 0:
            device1 += 1
        if row["defect_found_qty"] > 0 and row["judgment"] != "불합격":
            device3 += 1

    for ncr in tables["qms_nonconformance"]:
        disposition = disposition_by_ncr[ncr["ncr_id"]]
        inspection = inspection_by_id.get(ncr["inspection_id"])
        if (
            inspection
            and inspection["mes_process_result_id"] is not None
            and mes[inspection["mes_process_result_id"]]["result"] == "Fail"
            and disposition["disposition_type"] == "특채"
        ):
            device2 += 1
        if ncr["ncr_source"] == "입고검사" and ncr["lot_id"] in lots:
            device4 += 1

    for row in tables["qms_disposition"]:
        if row["rework_result"] == "실패" and row["disposition_type"] == "폐기":
            device5 += 1

    return dict(zip(DEVICE_TARGETS, (device1, device2, device3, device4, device5)))


In [ ]:
"""Spark 테이블 스키마와 전체 조립.

pyspark 를 로컬에서 import 할 수 없으므로 스키마를 DDL 문자열로 표현한다.
문자열은 pytest 에서 검증할 수 있고, Fabric 에서는
spark.createDataFrame(rows, schema=TABLE_DDL[name]) 로 그대로 쓰인다.
전부 None 인 컬럼이 있어 타입 추론에 맡길 수 없다.
"""

from __future__ import annotations


TABLE_DDL = {
    "qms_defect_code": (
        "defect_code STRING, defect_name_ko STRING, defect_name_en STRING, "
        "mes_defect_code STRING, defect_category STRING, severity STRING, "
        "severity_score INT, typical_step_codes STRING, standard_cause_ko STRING, "
        "standard_action_ko STRING, is_active BOOLEAN"
    ),
    "qms_inspection_spec": (
        "spec_id STRING, product_code STRING, product_name STRING, step_code STRING, "
        "step_name STRING, characteristic_code STRING, characteristic_name_ko STRING, "
        "measurement_type STRING, unit STRING, target_value DOUBLE, lsl DOUBLE, "
        "usl DOUBLE, cpk_target DOUBLE, sampling_method STRING, sample_size INT, "
        "inspection_frequency STRING, control_method_ko STRING, spec_version STRING, "
        "effective_from DATE, is_active BOOLEAN"
    ),
    "qms_inspector": (
        "inspector_id STRING, inspector_name STRING, team_ko STRING, shift_code STRING, "
        "qualification_level STRING, certified_characteristics STRING, "
        "certified_from DATE, certified_until DATE, is_active BOOLEAN"
    ),
    "qms_inspection": (
        "inspection_id STRING, inspection_type STRING, lot_id STRING, product_code STRING, "
        "product_name STRING, step_code STRING, step_name STRING, eqp_id STRING, "
        "mes_process_result_id INT, inspector_id STRING, inspection_datetime TIMESTAMP, "
        "sample_size INT, inspected_wafer_qty INT, judgment STRING, judgment_basis_ko STRING, "
        "defect_found_qty INT, measurement_count INT, has_nonconformance BOOLEAN, remark_ko STRING"
    ),
    "qms_measurement": (
        "measurement_id STRING, inspection_id STRING, spec_id STRING, lot_id STRING, "
        "product_code STRING, step_code STRING, characteristic_code STRING, "
        "characteristic_name_ko STRING, sample_no INT, site_no INT, measured_value DOUBLE, "
        "unit STRING, target_value DOUBLE, lsl DOUBLE, usl DOUBLE, deviation_pct DOUBLE, "
        "is_out_of_spec BOOLEAN, judgment STRING, metrology_eqp_id STRING, "
        "measured_by STRING, measured_at TIMESTAMP"
    ),
    "qms_incoming_inspection": (
        "iqc_id STRING, material_code STRING, material_name STRING, supplier_code STRING, "
        "supplier_name_ko STRING, supplier_lot_no STRING, receipt_date DATE, "
        "received_qty DOUBLE, uom STRING, sample_size INT, inspection_items_ko STRING, "
        "judgment STRING, defect_code STRING, coa_received BOOLEAN, coa_conformance STRING, "
        "inspector_id STRING, inspection_date DATE, remark_ko STRING"
    ),
    "qms_nonconformance": (
        "ncr_id STRING, ncr_source STRING, inspection_id STRING, iqc_id STRING, "
        "lot_id STRING, product_code STRING, product_name STRING, step_code STRING, "
        "step_name STRING, material_code STRING, eqp_id STRING, defect_code STRING, "
        "mes_defect_code STRING, severity STRING, affected_qty INT, detected_date DATE, "
        "root_cause_category STRING, root_cause_ko STRING, immediate_action_ko STRING, "
        "owner_dept_ko STRING, owner_name STRING, status STRING, due_date DATE, "
        "closed_date DATE, estimated_cost_krw BIGINT"
    ),
    "qms_disposition": (
        "disposition_id STRING, ncr_id STRING, lot_id STRING, product_code STRING, "
        "step_code STRING, disposition_type STRING, disposition_qty INT, decision_date DATE, "
        "decision_body_ko STRING, approver_name STRING, approval_status STRING, "
        "rework_step_code STRING, rework_result STRING, scrap_cost_krw BIGINT, "
        "effectiveness_check_date DATE, effectiveness_result STRING, reason_ko STRING"
    ),
}


def ddl_columns(ddl: str) -> tuple[str, ...]:
    return tuple(part.strip().split()[0] for part in ddl.split(","))


TABLE_COLUMNS = {name: ddl_columns(ddl) for name, ddl in TABLE_DDL.items()}


def build_all_tables(snapshot: MesSnapshot) -> dict[str, list[dict]]:
    """8개 테이블 전체를 만든다. 노트북과 테스트가 공유하는 단일 진입점이다."""
    inspectors = build_inspectors(snapshot)
    defect_codes = build_defect_codes()
    specs = build_inspection_specs(snapshot)
    inspections = build_inspections(snapshot, inspectors)
    measurements = build_measurements(inspections, specs, mes_anchor(snapshot))
    incoming = build_incoming_inspections(snapshot, inspectors, defect_codes)
    ncrs, dispositions = build_nonconformances(snapshot, inspections, incoming, defect_codes)
    return {
        "qms_defect_code": defect_codes,
        "qms_inspection_spec": specs,
        "qms_inspector": inspectors,
        "qms_inspection": inspections,
        "qms_measurement": measurements,
        "qms_incoming_inspection": incoming,
        "qms_nonconformance": ncrs,
        "qms_disposition": dispositions,
    }


def to_rows(table_name: str, rows: list[dict]) -> list[tuple]:
    """dict 를 컬럼 순서 tuple 로 바꾼다.

    spark.createDataFrame 에 dict 를 넘기면 키 순서 경고와 함께 스키마가
    알파벳순으로 재정렬된다. tuple 로 넘기면 DDL 순서가 그대로 지켜진다.
    """
    columns = TABLE_COLUMNS[table_name]
    return [tuple(row[column] for column in columns) for row in rows]


In [ ]:
# MES 연결 게이트. REST 와 MCP 두 채널이 모두 살아 있어야 진행합니다.
CLIENT = MesClient(MES_BASE_URL, MES_API_KEY)
try:
    SNAPSHOT = CLIENT.fetch_snapshot()
except Exception as exc:
    raise RuntimeError(
        "MES 연결에 실패했습니다. 확인할 것: "
        "(1) API 키가 올바른가 (2) Fabric Spark 풀에서 외부 인터넷 아웃바운드가 허용되는가 "
        f"(3) {MES_BASE_URL} 가 응답하는가. 원인: {exc}"
    ) from exc

print(
    f"제품 {len(SNAPSHOT.products)} · 자재 {len(SNAPSHOT.materials)} · BOM {len(SNAPSHOT.bom)} · "
    f"로트 {len(SNAPSHOT.lots)} · 공정이력 {len(SNAPSHOT.process_results)} · "
    f"라우트 {len(SNAPSHOT.route)} · 설비 {len(SNAPSHOT.equipment)}"
)


In [ ]:
TABLES = build_all_tables(SNAPSHOT)
for name, rows in TABLES.items():
    print(f"{name:26} {len(rows):5}행")
print(f"{'합계':24} {sum(len(rows) for rows in TABLES.values()):5}행")


In [ ]:
# 적재 전에 검증합니다. 이미 쓴 Delta 테이블은 되돌릴 수 없기 때문입니다.
RESULTS = validate(SNAPSHOT, TABLES)
print(format_report(RESULTS))
raise_on_fatal(RESULTS)


In [ ]:
# saveAsTable 은 홑이름이면 현재 카탈로그·스키마, 즉 Attach 한 레이크하우스에 씁니다.
# "레이크하우스이름.테이블" 같은 2단 이름은 스키마.테이블로 해석돼 실패합니다.
for name, rows in TABLES.items():
    assert name.startswith(TABLE_PREFIX), f"테이블 접두사 규칙 위반: {name}"
    target = f"{TARGET_SCHEMA}.{name}" if TARGET_SCHEMA else name
    frame = spark.createDataFrame(to_rows(name, rows), schema=TABLE_DDL[name])
    frame.write.format("delta").mode(WRITE_MODE).saveAsTable(target)
    print(f"{target:34} {frame.count():5}행 적재")


## 다음 단계

1. 레이크하우스에서 `qms_` 8개 테이블을 확인합니다.
2. 이 레이크하우스를 원본으로 Fabric Data Agent를 만들고, `data-agent-schema.md`의
   설명을 에이전트 지식으로 넣습니다.
3. Foundry 에이전트에 이 Data Agent와 MES MCP 엔드포인트를 함께 붙입니다.

## 두 시스템을 함께 봐야 답이 나오는 질문

- 생산은 통과했는데 품질이 잡아 세운 로트는 무엇이고 왜 그런가요? (3건)
- 생산에서 불합격인데 출하가 승인된 로트가 있나요? 승인 근거는요? (2건)
- 설비는 불량으로 판정하지 않았는데 검사에서 결함이 나온 건은요? (4건)
- 입고검사에서 불합격난 자재가 실제로 투입된 로트를 추적해 주세요. (2건)
- 재작업을 했지만 결국 폐기된 로트는요? (2건)
